# Módulo 3 · Clase 5 (Teoría) — De las RNN a los LLMs
### Deep Learning · Apunte de cátedra

**Objetivos.** Al terminar, los estudiantes deberían poder:
1. Representar texto como vectores (tokenización y embeddings).
2. Explicar cómo una RNN procesa secuencias y por qué sufre con dependencias largas (y cómo LSTM/GRU ayudan).
3. Entender el esquema seq2seq, su cuello de botella, y cómo la **atención** lo resuelve.
4. Describir el mecanismo de **self-attention** y la arquitectura **Transformer**.
5. Explicar qué es un **LLM**: pretraining autoregresivo, tokenización, las tres familias de arquitecturas, y fine-tuning / PEFT.

**Agenda (≈ 3 horas, con un descanso):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Del texto a vectores | 15 |
| 1 | Modelos de secuencias y RNN | 30 |
| 2 | Dependencias largas: LSTM/GRU | 20 |
| 3 | Seq2seq y el cuello de botella | 20 |
| — | *Descanso* | 15 |
| 4 | Atención neuronal | 25 |
| 5 | Self-attention y el Transformer | 35 |
| 6 | Modelos de lenguaje (LLMs) | 30 |


In [ ]:
# Setup
import numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
!pip install dlviz

## Bloque 0 · Del texto a vectores

Las redes operan sobre números, no sobre palabras. El pipeline de NLP empieza por convertir texto en vectores:

1. **Tokenización:** partir el texto en unidades (*tokens*): palabras, sub-palabras o caracteres.
2. **Vocabulario:** asignar un entero (id) a cada token.
3. **Embedding:** mapear cada id a un **vector denso** aprendible.

¿Por qué embeddings y no one-hot? Un one-hot es enorme (dimensión = tamaño del vocabulario) y no captura significado: todas las palabras quedan equidistantes. Un **embedding** es compacto y aprende geometría semántica (palabras parecidas → vectores cercanos).

In [ ]:
# Demo: tokenización simple + tabla de embeddings
frase = "el bebé está llorando".split()
vocab = {w:i for i,w in enumerate(["<pad>","el","bebé","está","llorando","perro"])}
ids = torch.tensor([vocab[w] for w in frase])
print("tokens:", frase)
print("ids:   ", ids.tolist())

emb = nn.Embedding(num_embeddings=len(vocab), embedding_dim=5)
vectores = emb(ids)
print("\nCada token -> vector denso de dim 5:")
print(vectores.detach().round(decimals=2))
print("\nLa tabla de embeddings es entrenable:", emb.weight.shape, "(vocab x dim)")

## Bloque 1 · Modelos de secuencias y RNN

El texto es una **secuencia**: el orden importa y la longitud varía. Distintas tareas combinan secuencias y vectores de distintas formas.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_00_tipos_rnn.png" width="760">

<sub>Tipos de tareas con secuencias: clasificación (seq→vec), generación (vec→seq), traducción y análisis (seq→seq).</sub>

Una **red recurrente (RNN)** procesa la secuencia **paso a paso**, manteniendo un **estado oculto** $h_t$ que resume lo visto hasta el momento:

$$ h_t = \tanh(W_{hh}\, h_{t-1} + W_{xh}\, x_t + b) $$

El mismo conjunto de pesos $(W_{hh}, W_{xh})$ se reutiliza en cada paso (pesos compartidos en el tiempo).

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_01_rnn_cell.png" width="620">

<sub>Celda RNN: combina la entrada actual x_t con el estado previo h_{t-1} para producir el nuevo estado h_t. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M3/01_RNN.png)</sub> 

In [ ]:
# Demo: una celda RNN a mano, desenrollada sobre una secuencia
torch.manual_seed(0)
dim_x, dim_h = 5, 8
Wxh = torch.randn(dim_h, dim_x)*0.3
Whh = torch.randn(dim_h, dim_h)*0.3
b   = torch.zeros(dim_h)

h = torch.zeros(dim_h)                 # estado inicial
secuencia = torch.randn(4, dim_x)      # 4 tokens (embeddings)
for t, x in enumerate(secuencia):
    h = torch.tanh(Wxh @ x + Whh @ h + b)
    print(f"paso {t}: ||h|| = {h.norm():.3f}  (el estado se actualiza con cada token)")

In [ ]:
# Demo: lo mismo con nn.RNN (procesa toda la secuencia de una)
rnn = nn.RNN(input_size=5, hidden_size=8, batch_first=True)
x = torch.randn(1, 4, 5)               # (batch, longitud, dim)
salidas, h_final = rnn(x)
print("salidas por paso:", salidas.shape, "| estado final:", h_final.shape)

In [ ]:
from dlviz import rnn_interactiva
rnn_interactiva()

## Bloque 2 · Dependencias largas: LSTM y GRU [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M3/02_LSTM.png)

Al desenrollar una RNN y entrenar con *backpropagation through time*, los gradientes se multiplican muchas veces → tienden a **desvanecerse** (o explotar). Resultado: la RNN "olvida" información lejana y le cuesta aprender **dependencias largas**.

Las **LSTM** (y su prima más simple, la **GRU**) resuelven esto con un **estado de celda** y **compuertas** (forget / input / output) que controlan qué información se conserva, se actualiza o se olvida. La celda da un "camino" por el que el gradiente fluye sin atenuarse.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_02_lstm.png" width="760">

<sub>LSTM: el estado de celda actúa como una autopista de memoria; las compuertas regulan el flujo. El gradiente se preserva a lo largo del tiempo.</sub>

In [ ]:
# Demo: nn.LSTM expone estado oculto Y estado de celda
lstm = nn.LSTM(input_size=5, hidden_size=8, batch_first=True)
x = torch.randn(1, 4, 5)
salidas, (h_n, c_n) = lstm(x)
print("salidas:", salidas.shape, "| h_n:", h_n.shape, "| c_n (estado de celda):", c_n.shape)

In [ ]:
from dlviz import lstm_interactiva
lstm_interactiva()

## Bloque 3 · Seq2seq y el cuello de botella

Para tareas como **traducción**, usamos una arquitectura **encoder-decoder** (seq2seq): un *encoder* RNN lee la frase de origen y la resume en un vector; un *decoder* RNN genera la traducción palabra por palabra a partir de ese vector.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_03_seq2seq.png" width="760">

<sub>Seq2seq: el encoder comprime toda la frase de origen en una representación, y el decoder genera la salida condicionada a ella.</sub>

**El problema:** comprimir *toda* una frase (larga, compleja) en un **único** vector es un cuello de botella. El decoder, al generar cada palabra, no puede "volver a mirar" partes específicas de la entrada. Esto motiva la atención.

In [ ]:
from dlviz import seq2seq_interactiva
seq2seq_interactiva()

## Bloque 4 · Atención neuronal

**Idea (Bahdanau, 2014):** en cada paso de generación, el decoder **mira todos** los estados del encoder y decide a cuáles prestar atención, en lugar de depender de un solo vector resumen.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_04_atencion.png" width="760">

<sub>Atención: el decoder calcula scores sobre cada estado del encoder, los normaliza (softmax) y produce una suma ponderada.</sub>

Mecánica: se calcula un *score* entre el estado del decoder y cada estado del encoder; softmax convierte los scores en pesos que suman 1; la salida es la suma ponderada de los estados del encoder (el *contexto*).

Un beneficio adicional: los pesos de atención son **interpretables**. Al traducir, muestran qué palabra de origen se alinea con cada palabra generada.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_05_alignment.png" width="460">

<sub>Matriz de alineamiento: los pesos de atención revelan qué palabras de origen y destino se corresponden.</sub>

In [ ]:
# Demo: atención como suma ponderada (decoder mira 4 estados del encoder)
torch.manual_seed(1)
estados_encoder = torch.randn(4, 6)        # 4 tokens de origen, dim 6
estado_decoder  = torch.randn(6)           # estado actual del decoder
scores  = estados_encoder @ estado_decoder # similitud con cada estado
pesos   = torch.softmax(scores, dim=0)
contexto = pesos @ estados_encoder         # suma ponderada
print("pesos de atención (suman 1):", pesos.round(decimals=2).tolist())
print("-> el decoder se concentra en el token", pesos.argmax().item())

In [ ]:
from dlviz import nmt_atencion_interactiva
nmt_atencion_interactiva()

## Bloque 5 · Self-attention y el Transformer

Los Transformers (2017) llevan la atención al extremo: **eliminan la recurrencia**. En lugar de procesar token a token, cada token atiende **a todos los demás de la misma secuencia** (self-attention), y todo se computa en paralelo.

Cada token genera tres vectores mediante proyecciones aprendidas: **query** (qué busco), **key** (qué ofrezco) y **value** (qué entrego).

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_06_selfattention.png" width="760">

<sub>Self-attention: a partir de X se calculan Q, K y V con matrices aprendidas. H = softmax(QKᵀ/√dₖ)·V.</sub>

$$ \text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V $$

$QK^\top$ mide la similitud entre cada par de tokens; softmax la convierte en pesos; el resultado es un promedio ponderado de los *values*.

In [ ]:
# Demo: self-attention desde cero, visualizando la matriz de atención
torch.manual_seed(0)
tokens = ["el","bebé","está","llorando"]
n, d, dk = len(tokens), 16, 16
X = torch.randn(n, d)
Wq, Wk, Wv = (torch.randn(d, dk) for _ in range(3))
Q, K, V = X@Wq, X@Wk, X@Wv
A = torch.softmax(Q@K.T / dk**0.5, dim=-1)   # matriz de atención (n x n)
out = A @ V

plt.figure(figsize=(4.5,4))
plt.imshow(A.detach(), cmap='viridis')
plt.xticks(range(n), tokens, rotation=45); plt.yticks(range(n), tokens)
plt.xlabel("atiende a"); plt.ylabel("token"); plt.title("Matriz de self-attention"); plt.colorbar(); plt.show()
print("Cada fila suma 1:", A.sum(-1).round(decimals=2).tolist())

In [ ]:
from dlviz import self_attention_interactiva
self_attention_interactiva()

**Multi-head:** en vez de una sola atención, se usan varias "cabezas" en paralelo, cada una capaz de capturar un tipo de relación distinto. Sus salidas se concatenan.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_07_multihead.png" width="760">

<sub>Multi-head self-attention: varias cabezas en paralelo; cada una aprende a fijarse en relaciones distintas.</sub>

In [ ]:
from dlviz import multi_head_interactiva
multi_head_interactiva()

El **Transformer** completo apila bloques de (multi-head attention + MLP + normalización + conexiones residuales). Existe en variante encoder-decoder (traducción) o solo-encoder / solo-decoder.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_08_transformer.png" width="560">

<sub>Arquitectura Transformer: bloques de atención multi-cabeza y MLP, con normalización y conexiones residuales, repetidos N veces.</sub>

Dos piezas clave:
- **Positional encoding:** como no hay recurrencia, el Transformer no conoce el orden. Se *suma* a cada embedding una codificación de posición (típicamente senoidal).
- **Máscara causal:** para *generar* texto, cada token solo puede atender a los **anteriores** (no al futuro). Se logra poniendo $-\infty$ en los scores de las posiciones futuras antes del softmax.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_09_causal_mask.png" width="560">

<sub>Máscara causal: en generación, un token solo atiende al pasado. Las posiciones futuras se anulan (softmax → 0).</sub>

In [ ]:
from dlviz import positional_encoding_interactiva
positional_encoding_interactiva()

In [ ]:
# Demo: positional encoding senoidal
def positional_encoding(seq_len, d):
    pos = np.arange(seq_len)[:,None]; i = np.arange(d)[None,:]
    angle = pos / np.power(10000, (2*(i//2))/d)
    pe = np.zeros((seq_len, d)); pe[:,0::2]=np.sin(angle[:,0::2]); pe[:,1::2]=np.cos(angle[:,1::2])
    return pe
plt.figure(figsize=(7,3))
plt.imshow(positional_encoding(50, 64).T, cmap='RdBu', aspect='auto')
plt.xlabel("posición en la secuencia"); plt.ylabel("dimensión"); plt.title("Positional encoding senoidal"); plt.colorbar(); plt.show()

In [ ]:
# Demo: máscara causal sobre una matriz de atención
n = 5
scores = torch.randn(n, n)
mask = torch.triu(torch.ones(n, n), diagonal=1).bool()   # futuro = True
scores_masked = scores.masked_fill(mask, float('-inf'))
A = torch.softmax(scores_masked, dim=-1)
print("Matriz de atención causal (triangular inferior; el futuro queda en 0):")
print(A.round(decimals=2))

## Bloque 6 · Modelos de lenguaje (LLMs)

Un **modelo de lenguaje** asigna probabilidades a secuencias de tokens. Los LLMs modernos son Transformers gigantes entrenados sobre cantidades enormes de texto con un objetivo sorprendentemente simple: **predecir el siguiente token**.

La gran idea: *muchísimas* tareas (resumir, traducir, responder, programar) pueden plantearse como "predecir la siguiente palabra" dado un contexto. Entrenar a gran escala en este objetivo produce modelos sorprendentemente capaces.

### Tres familias de arquitecturas

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_11_llm_arquitecturas.png" width="760">

<sub>Tres familias: solo-decoder (GPT, Claude, Llama; generativos), solo-encoder (familia BERT; comprensión) y encoder-decoder (Flan-T5, Whisper).</sub>

- **Solo-decoder (autoregresivos):** GPT, Llama, Claude. Generan texto de izquierda a derecha (con máscara causal). Son los "LLMs generativos".
- **Solo-encoder:** familia **BERT**. Ven toda la secuencia a la vez; ideales para *comprensión* (clasificación, NER). Se preentrenan enmascarando palabras (*masked language modeling*).
- **Encoder-decoder:** Flan-T5, Whisper. Para tareas de secuencia-a-secuencia.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_10_bert.png" width="620">

<sub>BERT (solo-encoder): se preentrena prediciendo palabras enmascaradas y luego se hace fine-tuning para tareas específicas. [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M3/03_BERT.png)</sub>

### Adaptar un LLM: fine-tuning y PEFT [Ver infografía](https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/infografias/M3/04_LLM.png)
Un modelo preentrenado se **adapta** a nuestra tarea. El fine-tuning completo reentrena *todos* los pesos: potente pero costoso (un modelo por tarea).

**PEFT (Parameter-Efficient Fine-Tuning)** ajusta solo una fracción de parámetros. El método más popular es **LoRA**: congela los pesos $W$ y aprende una corrección de **rango bajo** $A\cdot B$ (con $A$ y $B$ pequeñas). La salida pasa de $h = xW$ a $h = xW + xAB$.

<img src="https://raw.githubusercontent.com/ivansipiran/Deep-Learning/main/img/m3_12_lora.png" width="620">

<sub>LoRA: se congelan los pesos preentrenados W y se entrena solo una descomposición de bajo rango A·B. Muchos menos parámetros que ajustar.</sub>

## Cierre

Recorrimos el camino del NLP moderno: de representar texto con embeddings, a las RNN y su límite con dependencias largas, al seq2seq con atención, hasta los Transformers y los LLMs.

**En la clase práctica (Clase 6)** van a tokenizar texto, entrenar una RNN/LSTM para clasificar sentimiento, hacer fine-tuning de un modelo tipo BERT con Hugging Face, y usar un LLM generativo.

**En el Módulo 4** veremos modelos generativos, multimodales y agentes.